In [1]:
import librosa
import torch
import torch.nn.functional as F
import os
import numpy as np
from pathlib import Path
import copy # for deepcopy
import time

import matplotlib.pyplot as plt
from IPython.display import Audio, display

from transformers import EncodecModel, AutoProcessor

from realtime_synth_ui import build_synth_ui # pip install "rtpysynth[ui] @ git+https://github.com/lonce/RTPySynth@v0.1.4"

In [2]:
import time

# NEW: required import for threaded version
from concurrent.futures import ThreadPoolExecutor

In [3]:
# for the rt synth
from realtime_synth.generators.base import BaseGenerator
from realtime_synth.utils import exp_map01
from realtime_synth_ui import build_synth_ui

# import the system demo synths just to have them on the interface
from realtime_synth.generators.sine import SineGenerator
from realtime_synth.generators.noisy_lp import NoisyLPGenerator

In [4]:
# for RNN4Control
from model.gru_audio_model import RNN, GRUModelConfig
from audioDataLoader.audio_dataset import  efficient_codes_to_latents, preprocess_latents_for_RNN # , latents_to_audio_simple,

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Synth/Encodec Parameters</b>

In [5]:
#audiopath = data_dir / "DSBugs--busybodyFreqFactor-00.60--c-00--x-00.wav"
# -- kbs is a training and encoding parameter only!
#Kbs = 3  # Supported bandwidths are 1.5kbps (n_q = 2), 3 kbps (n_q = 4), 6 kbps (n_q = 8) and 12 kbps (n_q =16) and 24kbps (n_q=32).
buffersize = 320 #[NOTE - not tested on values other than 24000/75 - the frame length of encodec codes in samples]

g_hopsize=12
g_chunksize=25
sr=24000

offline_render_duration=20 #seconds
offline_render_frames=offline_render_duration*75

SYNTEX=1

if SYNTEX: # syntex
    g_param_labels = ["param 1", "Amp"]
    g_norm_param_vals = [.5, .5]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=[.5]
else: #nsynth
    g_param_labels = ["class", "pitch", "amp"]
    g_norm_param_vals = [0, .5, .5]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=[0, .5, .5]          # for the neural network

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>RNN Parameters</b>

In [6]:
# Where is the checkpoint directory?
#run_directory = str(Path('./output/20250828_173834_encodectest_DSWind')) # 'Path to the directory of the saved run.'
#run_directory = str(Path('./output/20250827_164925_bees_good')) # 'Path to the directory of the saved run.'
#run_directory = str(Path('./output/20250829_113424_encodectest_ChirpPattern')) # 'Path to the directory of the saved run.' 
#run_directory = str(Path('./output/20250905_130844_encodectest_DSWind_splitseqs')) # 'Path to the directory of the saved run.'
#run_directory = str(Path('./output/20250905_153959_encodectest_TokWotalDuet_splitseqs_sl125_nodes64_6l')) # 'Path to the directory of the saved run.'
#run_directory = str(Path('./output/20250906_170007_encodectest_DSPistons_splitseqs_sl125_3l_size64_nq4'))
#run_directory = str(Path('./output/20250906_210430_encodectest_TokWotalDuet_splitseqs_sl125_3l_size64_nq8_emb4.5'))
#run_directory = str(Path('./output/20250911_174140_encodectest_TokWotalDuet_splitseqs_sl125_3l_size128_nq8_emb4.5_reverseQweights'))
#run_directory = str(Path('./output/20250912_171846_DSPistons_sl125_3l_size128_nq4_emb4.5_seqYesTF'))
run_directory = str(Path('./output/20250918_154544_DSPistons_TF'))
run_directory = str(Path('./output/20250918_184608_DSPistons_PARALLEL'))
checkpoint_fname =   "last_checkpoint.pt" # "checkpoint_150.pt" # 

g_top_n = 16 #'Sample from the top N most likely outputs.'
g_temperature = .8 #'Controls the randomness of predictions.'

frame_rate=75
device='cpu'

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>RNN CLASS</b>

In [7]:
class RNNGenerator():
    def __init__(self, checkpoint_path, model_config, data_config, enc_model, chunksize, hopsize, init_cond, top_n, temperature) : 
        #self.chkpt = chkpt

        self.clamp_val = data_config.clamp_val # need this to map between encodec latents and model input ranges

        self.model = RNN(model_config).to(device)
        checkpoint = torch.load(checkpoint_path, map_location=device)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model.eval()

        self.enc_model=enc_model

        self.codebook_size = self.model.config.codebook_size
        self.n_q = self.model.config.n_q
        self.cond_size = self.model.config.cond_size
    

        self.chunksize=chunksize
        self.hopsize=hopsize

        self.top_n=top_n
        self.temperature=temperature

        #state between call to generate steps
        self.hidden = None # updated on every sequence step in warmup and in run_inference
        
        self.current_latent = self.warmup_rnn(init_cond) # initializes self.current_latent

        #initialize the "current_code_chunk" so we have the history window ready for the first and following getNextCodeChunk calls
        self.current_code_chunk = self.run_inference(init_cond, self.chunksize) #TIME should be the last dimension
        

    def warmup_rnn(self, init_cond, warmup_len=10) :
        sd=.33 # to create data in [-1,1
        warmup_latents = warmup_tensor = torch.clamp(torch.randn(warmup_len, 128) * sd, -3*sd, 3*sd)  

            # Ensure warmup_latents is on the right device and has correct shape
        warmup_latents = warmup_latents.to(device)  # Shape: (warmup_len, 128)
        #warmup_len = warmup_latents.shape[0]
    
        # Handle conditioning
        if self.cond_size > 0 and init_cond is not None:
            init_cond_tensor = torch.tensor(init_cond)
            # Use first conditioning vector for entire warmup
            first_cond_vec = init_cond_tensor.unsqueeze(0).repeat(warmup_len, 1).to(device)  # (warmup_len, cond_size)
            warmup_full_input = torch.cat([warmup_latents, first_cond_vec], dim=-1)  # (warmup_len, 128 + cond_size)
        else:
            # No conditioning
            warmup_full_input = warmup_latents  # (warmup_len, 128)
        
        self.hidden = self.model.init_hidden(batch_size=1)
        # This propogates hidden state, but doesn't bring the output back down for the next input, using the warmup vectors instead.
        for i in range(len(warmup_full_input)):
            _, self.hidden = self.model(warmup_full_input[i].unsqueeze(0), self.hidden, encodec_model=self.enc_model, batch_size=1)
        
        # Get the last latent for starting generation
        return warmup_latents[-1].unsqueeze(0)  # (1, 128)


        
    #return the next hopsize codes
    def run_inference(self, params, generation_length) :
        generated_codes = []
        with torch.no_grad():
            for i in range(generation_length):
                # Handle conditioning for this step
                if self.cond_size > 0 and params is not None:
                    current_cond_vec = torch.tensor(params).unsqueeze(0).to(device)  # (1, cond_size)
                    next_input_full = torch.cat([self.current_latent, current_cond_vec], dim=-1)  # (1, 128 + cond_size)
                else:
                    # No conditioning
                    next_input_full = self.current_latent  # (1, 128)
    
                logits_list, self.hidden = self.model(next_input_full, self.hidden, encodec_model=self.enc_model, batch_size=1)
    
                # Transform outputs to next input using the extracted function
                self.current_latent, sampled_codes = self.transform_outputs_to_inputs(logits_list)
                generated_codes.append(sampled_codes)
        return np.array(generated_codes).T.tolist()  # make time the last dimension for the encodec decoder
        
        
    def getNextCodeChunk(self, params, generation_length) :
        # Generate new hopsize codes
        hopcodes = self.run_inference(params, generation_length)
        # push them on to the end of current_code_chunk
        self.current_code_chunk = [(row + b_row)[generation_length:] for row, b_row in zip(self.current_code_chunk, hopcodes)]
        # return the new chunk
        return self.current_code_chunk     #and return it
        
    def transform_outputs_to_inputs(self, logits_list):
        """
        Transform model outputs (logits) into the next input (128D latent).
        
        Args:
            logits_list: List of logit tensors, one per quantizer
        
        Returns:
            torch.Tensor: Next input latent of shape (1, 128)
        """
        device = logits_list[0].device
        self.enc_model.to(device)
        sampled_codes = []
        
        for j in range(self.n_q):
            # Apply temperature and get top-k
            logits_j = logits_list[j].div(self.temperature).squeeze()  # (codebook_size,)
            top_n_logits, top_n_indices = torch.topk(logits_j, self.top_n)
            top_n_probs = F.softmax(top_n_logits, dim=-1)
            
            # Sample from top-k
            try:
                sampled_relative_idx = torch.multinomial(top_n_probs, 1).squeeze()
                sampled_code = top_n_indices[sampled_relative_idx]
                sampled_codes.append(sampled_code.item())
            except Exception as e:
                print(f"Sampling error for quantizer {j}: {e}")
                # Fallback to random sampling
                sampled_codes.append(torch.randint(0, self.codebook_size, (1,)).item())
        
        # Convert sampled codes back to latent - CREATE TENSOR ON CORRECT DEVICE
        codes_tensor = torch.tensor(sampled_codes, device=device).unsqueeze(0).unsqueeze(-1)  # (1, n_q, 1)
        next_latent = efficient_codes_to_latents(self.enc_model, codes_tensor).squeeze(0).squeeze(-1).unsqueeze(0)  # (1, 128)
        next_latent = preprocess_latents_for_RNN(next_latent, self.clamp_val)
        
        return next_latent, sampled_codes    

In [8]:
# load the encoder model 
#####################################################################
enc_model = EncodecModel.from_pretrained("facebook/encodec_24khz")
enc_model.eval()

# --- these are only for encoding, not decoding
#enc_model.config.target_bandwidths = [Kbs] # Supported bandwidths are 1.5kbps (n_q = 2), 3 kbps (n_q = 4), 6 kbps (n_q = 8) and 12 kbps (n_q =16) and 24kbps (n_q=32).
#processor = AutoProcessor.from_pretrained("facebook/encodec_24khz", use_fast=False)
enc_model.device

device(type='cpu')

In [9]:
# load the RNN model 
#####################################################################
config_path = os.path.join(run_directory, "config.pt")
checkpoint_path = os.path.join(run_directory, "checkpoints", checkpoint_fname) 

assert os.path.exists(run_directory), f"Run directory not found: {run_directory}"
assert os.path.exists(config_path), f"Config file not found: {config_path}"
assert os.path.exists(checkpoint_path), f"Checkpoint file not found: {checkpoint_path}"

saved_configs = torch.load(config_path, weights_only=False)
model_config = saved_configs["model_config"]
data_config = saved_configs["data_config"]

rnngen = RNNGenerator(checkpoint_path, model_config, data_config, enc_model, g_chunksize, g_hopsize, g_init_cond, g_top_n, g_temperature)
print("Model successfully loaded from checkpoint.")
print(f"Using device = {device}")
rnngen.clamp_val 

Latents embedded in 64 of the GRU input size of 128
Conditioning parameters embedded in 64 of the GRU input size of 128
Model successfully loaded from checkpoint.
Using device = cpu


15

In [10]:
# print(f"rnngen.current_code_chunk is {rnngen.current_code_chunk}")
# rnngen.current_latent

In [11]:
# newfoo = rnngen.getNextCodeChunk(g_init_cond, g_hopsize)
# print(f"rnngen.current_code_chunk is {rnngen.current_code_chunk}")
# rnngen.current_latent

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>CodecSynth</b>

In [12]:

class MyEncodecPlayer(BaseGenerator):
    # normalized params in [0,1]
    param_labels = g_param_labels

    # -----------------------
    def __init__(self, rnngen, buffersize, init_norm_params=None):
        super().__init__(init_norm_params or [0.5, 0.6])  # defaults
        self.set_params(self.norm_params)  # initialize semantic values

        self.rnngen = rnngen
        self.cond_size = rnngen.cond_size

        self.chunksizeframes = g_chunksize   # decode this many frames each time
        self.framehopsize    = g_hopsize     # decode a new chunk every framehopsize
        self.nextendframe    = self.framehopsize

        self.buffersize = buffersize
        self.nextsample = 0
        # NOTE: assumes global sr and frame_rate are defined elsewhere in your code
        self.framesizesamples = sr // frame_rate  # e.g., 75; encoder is 75 fps

        self.currentchunkframe = 0  # nth frame in the chunk of audio we are playing
        self.seeding_len = self.chunksizeframes - self.framehopsize
        self.genaudioframe = 0      # mth frame we've generated in total

        self._last_error = None
        self._decodetime = 0.0
        self._callrecord = ""

        # small scratch buffer to avoid per-callback allocations (optional)
        self._scratch = np.empty(self.buffersize, dtype=np.float32)

        # === Preload first hop synchronously (unchanged behavior) ===
        with torch.inference_mode():
            FOO = rnngen.getNextCodeChunk(self.norm_params[:self.cond_size], self.framehopsize)
            self.thisaudioseq = enc_model.decode(
                torch.tensor(FOO).unsqueeze(0).unsqueeze(0), [None], None
            )[0]
        self.thisaudioseq = self.thisaudioseq[0, 0].detach().numpy()
        self.thisaudioseq = self.thisaudioseq[-self.framehopsize * self.framesizesamples:]
        self.nextaudioseq = None  # will be filled by background worker

        # NEW: single background worker + a future for the next hop
        self._hop_exec = ThreadPoolExecutor(max_workers=1, thread_name_prefix="HopGen")
        self._next_future = None

        # Kick off the first async hop immediately
        self._schedule_next_hop()

    # -----------------------
    def _schedule_next_hop(self):
        """Launch getNextAudioHop() in the background (non-blocking)."""
        if self._next_future is None:
            try:
                self._next_future = self._hop_exec.submit(self.getNextAudioHop)
            except Exception as e:
                self._last_error = f"scheduling error: {e!r}"
                self._next_future = None

    # -----------------------
    def _try_collect_next(self):
        """
        If the background hop has finished, collect it into self.nextaudioseq (non-blocking).
        """
        fut = self._next_future
        if fut is not None and fut.done():
            try:
                self.nextaudioseq = fut.result()
            except Exception as e:
                self._last_error = f"hop result error: {e!r}"
                self.nextaudioseq = None
            finally:
                self._next_future = None  # allow scheduling the following hop

    # -----------------------
    def close(self):
        """Optional: call when tearing down to stop the worker quickly."""
        try:
            if hasattr(self, "_hop_exec") and self._hop_exec:
                self._hop_exec.shutdown(wait=False, cancel_futures=True)
        except Exception:
            pass

    # -----------------------
    def getNextAudioHop(self):
        """
        Heavy work: RNN inference + EnCodec decode for one hop.
        Runs on the background thread.
        Returns a 1D numpy array of length framehopsize * framesizesamples (mono, float32 or convertible).
        """
        self.genaudioframe = self.genaudioframe + self.framehopsize
        self._callrecord = self._callrecord + f";(start: {self.genaudioframe}, end: {self.genaudioframe + self.chunksizeframes})"

        start_time = time.monotonic()
        with torch.inference_mode():
            next_codes = self.rnngen.getNextCodeChunk(self.norm_params[:self.cond_size], self.framehopsize)
            nextseq = enc_model.decode(torch.tensor(next_codes).unsqueeze(0).unsqueeze(0), [None], None)[0]
        nextseq = nextseq[0, 0].detach().numpy()
        self._decodetime += (time.monotonic() - start_time)

        # take only the hopsize of audio that we need
        return nextseq[-self.framehopsize * self.framesizesamples:]

    # -----------------------
    # This just sets values used for disaply (the norm_params are the ones sent to the synth)
    def set_params(self, norm_params):
        super().set_params(norm_params)
        # Map [0,1] → semantic values (your mapping)
        # self.freq = float(exp_map01(self.norm_params[0], 20.0, 2000.0))  # exponential Hz
        # self.amp  = float(self.norm_params[1])                           # linear gain 0..1

        if SYNTEX: # syntex
            self.p0 = self.norm_params[0]
            self.p1 = self.norm_params[1]
        else: #nsynth
            self.p0 = self.norm_params[0]
            self.p1 = 64 + self.norm_params[1]*12
            self.p2 = self.norm_params[2]
        

    # -----------------------
    def generate(self, frames, sr):
        assert frames == self.buffersize, "ooh, you're in trouble if frames requested is different than the buffer size."

        # if self.amp <= 0.0 or self.freq <= 0.0:
        #     self._scratch.fill(0.0)
        #     return self._scratch

        # slice current hop
        endsamp = self.nextsample + self.buffersize
        y = self.thisaudioseq[self.nextsample:endsamp]
        self.nextsample = endsamp

        # NON-BLOCKING: if we just started a hop, see if the background result is ready
        if self.currentchunkframe == 0:
            self._try_collect_next()
            # if nothing in-flight, (re)start background worker
            if self._next_future is None:
                self._schedule_next_hop()

        # advance within hop; at hop boundary, try to swap
        self.currentchunkframe += 1
        if self.currentchunkframe == self.framehopsize:
            if self.nextaudioseq is not None:
                # swap in new hop (no copy; ensure float32)
                self.thisaudioseq = np.asarray(self.nextaudioseq, dtype=np.float32)
                self.nextaudioseq = None
                self._schedule_next_hop()  # immediately start computing the following hop
            else:
                # no hop ready → output SILENCE for one hop (your preference)
                msg = "missed hop swap"
                self._last_error = (self._last_error + " | " + msg) if self._last_error else msg
                self.thisaudioseq = np.zeros(self.framehopsize * self.framesizesamples, dtype=np.float32)
                # also (re)schedule next hop in case worker died
                if self._next_future is None:
                    self._schedule_next_hop()
            # reset for new hop window
            self.currentchunkframe = 0
            self.nextsample = 0

        # # Do post signal processing based on params if you need to 
        # # scale into scratch (avoids alloc every block)
        # np.multiply(y, self.amp, out=self._scratch, casting='unsafe')
        # return self._scratch

        #otherwise, just return the buffer
        return y

    # -----------------------
    def formatted_readouts(self):
        # Optional: pretty labels shown next to sliders
        if SYNTEX: # syntex
            return [
                # f"{self.param_labels[0]}: {self.freq:7.2f} Hz",
                # f"{self.param_labels[1]}: {self.amp:.3f}",
                f"{self.param_labels[0]}: {self.p0:1.1f} ",
                f"{self.param_labels[1]}: {self.p1:1.1f}"
            ]

        else: # nsynth
            return [
                # f"{self.param_labels[0]}: {self.freq:7.2f} Hz",
                # f"{self.param_labels[1]}: {self.amp:.3f}",
                f"{self.param_labels[0]}: {self.p0:1.1f} Hz",
                f"{self.param_labels[1]}: {self.p1:1.1f}",
                f"{self.param_labels[0]}: {self.amp:1.1f} Hz"               
                ]


<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>ON LINE, Realtime TEST</b>

In [13]:
GENS = {
    "MyEncodecPlayer": lambda: MyEncodecPlayer(rnngen, buffersize, g_norm_param_vals),
    "Sine": SineGenerator,            # defined in the default system
    "Noisy LP": NoisyLPGenerator,     # defined in the default system
}
print(f"Will use samplerate = {sr} and blocksize = {buffersize}")
synth, ui = build_synth_ui(GENS, samplerate=sr, blocksize=buffersize, channels=1)

Will use samplerate = 24000 and blocksize = 320


HTML(value='')

In [14]:
print(getattr(synth.gen, "_last_error", None))

None


In [15]:
print(getattr(synth.gen, "_callrecord ", None))

None


In [16]:
print(getattr(synth.gen, "__decodetime", None))

None


<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>OFF LINE TEST</b>

In [17]:
# just checking, not passed to to the synth which takes classes
foo=MyEncodecPlayer(rnngen, buffersize, g_norm_param_vals)
foo.thisaudioseq
len(foo.thisaudioseq)
#display(Audio(foo.thisaudioseq, rate=sr))

3840

In [18]:
# chunk=foo.thisaudioseq
# for i in range(0,1) :
#     foo.getNextAudioHop()
#     chunk = np.concatenate((chunk, foo.thisaudioseq), axis=0) 
# len(chunk)

In [19]:
c=[]

hops=int(offline_render_duration*frame_rate/g_hopsize)

for hopnum in range(0,hops) :
    c=  np.concatenate((c , foo.thisaudioseq), axis=0) 
    foo.thisaudioseq=foo.getNextAudioHop()
print(f"time spent decoding {offline_render_duration} secs of audio = {foo._decodetime:.2f}")


time spent decoding 20 secs of audio = 5.65


In [20]:
display(Audio(c, rate=sr))

In [21]:
print(f"{foo._callrecord}")

;(start: 12, end: 37);(start: 24, end: 49);(start: 36, end: 61);(start: 48, end: 73);(start: 60, end: 85);(start: 72, end: 97);(start: 84, end: 109);(start: 96, end: 121);(start: 108, end: 133);(start: 120, end: 145);(start: 132, end: 157);(start: 144, end: 169);(start: 156, end: 181);(start: 168, end: 193);(start: 180, end: 205);(start: 192, end: 217);(start: 204, end: 229);(start: 216, end: 241);(start: 228, end: 253);(start: 240, end: 265);(start: 252, end: 277);(start: 264, end: 289);(start: 276, end: 301);(start: 288, end: 313);(start: 300, end: 325);(start: 312, end: 337);(start: 324, end: 349);(start: 336, end: 361);(start: 348, end: 373);(start: 360, end: 385);(start: 372, end: 397);(start: 384, end: 409);(start: 396, end: 421);(start: 408, end: 433);(start: 420, end: 445);(start: 432, end: 457);(start: 444, end: 469);(start: 456, end: 481);(start: 468, end: 493);(start: 480, end: 505);(start: 492, end: 517);(start: 504, end: 529);(start: 516, end: 541);(start: 528, end: 553);(

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>OFF LINE TEST 2 - calling generate</b>

In [22]:
bar= MyEncodecPlayer(rnngen, buffersize, g_norm_param_vals)

In [23]:
#you can't just call generate in a loop like this because the calls happen way to fast for the NN to meet the needs of the fill buffer callback
# for buffernum in range(0,1375) :
#     audio_out =  np.concatenate((audio_out, bar.generate(buffersize, sr)), axis=0) 
    

In [24]:
#This function uses generate (but waits for the "next hop" generation from the neural net so it doesn't lose any audio.
# You can think of this as rendering as fast as it can - faster than real time, but only as fast as the NN can produce the data
#
def render_offline(player, n_blocks, sr, buffersize):
    blocks = []
    for i in range(n_blocks):
        # if we’re at the hop boundary, ensure next hop is ready
        if player.currentchunkframe == 0:
            # make sure a future is scheduled
            if player._next_future is None:
                player._schedule_next_hop()
            # wait here until it finishes, then collect
            if player._next_future is not None:
                player._next_future.result()   # blocks here offline
                player._try_collect_next()
        blocks.append(player.generate(buffersize, sr).copy())
    return np.concatenate(blocks).astype(np.float32)

# usage:
#audio_out = render_offline(bar, 1375, sr, buffersize)

In [25]:

start_time = time.monotonic()
audio_out = render_offline(bar, offline_render_frames, sr, buffersize)
rnnelapsed_time = time.monotonic() - start_time
print(f"RNN time to generate {offline_render_duration} secs of sound: {rnnelapsed_time:.2f}. Converting to audio...")


RNN time to generate 20 secs of sound: 5.42. Converting to audio...


In [26]:
display(Audio(audio_out, rate=sr)) 